In [ ]:
#Assignment 1
#NAME:Ram Kumar Singh
#Roll Number: 2501940002
#Course Code & Title: Deep Learning Architectures and Techniques(ETMMDL274)
#MCA(AI&ML)

In [ ]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
#IMPORTS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from sklearn.neural_network import MLPClassifier

In [ ]:
# 1. REPRODUCIBILITY

SEED = 42
np.random.seed(SEED)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2. LOAD DATASET
print("=" * 60)
print("  STEP 1: Loading Dataset")
print("=" * 60)

# Place Loan_default.csv in the same folder as this script
df = pd.read_csv("Loan_default.csv")

print(f"Dataset shape  : {df.shape}")
print(f"Columns        : {df.columns.tolist()}")
print(f"\nMissing values : {df.isnull().sum().sum()}")
print(f"\nTarget distribution:\n{df['Default'].value_counts()}")
print(f"Default rate   : {df['Default'].mean() * 100:.2f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3. DATA PREPROCESSING
print("\n" + "=" * 60)
print("  STEP 2: Data Preprocessing")
print("=" * 60)

# 3a. Drop non-informative ID column
df.drop(columns=["LoanID"], inplace=True)

# 3b. Separate features (X) and target (y)
X = df.drop(columns=["Default"])
y = df["Default"]

# 3c. Define feature groups
cat_cols = [
    "Education", "EmploymentType", "MaritalStatus",
    "HasMortgage", "HasDependents", "LoanPurpose", "HasCoSigner",
]
num_cols = [
    "Age", "Income", "LoanAmount", "CreditScore",
    "MonthsEmployed", "NumCreditLines", "InterestRate",
    "LoanTerm", "DTIRatio",
]

# 3d. Label-encode all categorical features
le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))
    print(f"  [Encoded] {col}")

# 3e. Train / Validation / Test split  (70% / 15% / 15%, stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"\n  Train size      : {X_train.shape[0]:>7,}")
print(f"  Validation size : {X_val.shape[0]:>7,}")
print(f"  Test size       : {X_test.shape[0]:>7,}")

# 3f. Standardise features (fit ONLY on training data to avoid leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print("\n  Features standardised (zero mean, unit variance).")

In [ ]:
# 4. MLP ARCHITECTURE & TRAINING

print("\n" + "=" * 60)
print("  STEP 3: MLP Architecture & Training")
print("=" * 60)
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=50,
    random_state=SEED,
    verbose=False,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=10,
    tol=1e-4,
)

print("\n  Architecture : Input(16) → Dense(64, ReLU) → Dense(32, ReLU) → Output(1, Sigmoid)")
print("  Optimizer    : Adam  |  lr = 0.001")
print("  Loss         : Binary Cross-Entropy")
print("  Batch size   : 256")
print("  Max epochs   : 50  (early stopping, patience = 10)")

print("\n  Training ...")
mlp.fit(X_train, y_train)
print(f"  Training stopped at epoch {mlp.n_iter_}")

In [ ]:
# 5. TRAINING CURVES
print("\n" + "=" * 60)
print("  STEP 4: Training Curves")
print("=" * 60)

train_loss  = mlp.loss_curve_
val_acc     = mlp.validation_scores_   # validation accuracy per epoch
epochs_axis = range(1, len(train_loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "MLP Training Curves – Loan Default Prediction",
    fontsize=14, fontweight="bold"
)

# --- Loss curve ---
axes[0].plot(epochs_axis, train_loss, "b-o", markersize=3, label="Training Loss")
axes[0].set_title("Loss Curve (Binary Cross-Entropy)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Validation accuracy curve ---
axes[1].plot(epochs_axis, val_acc, "g-o", markersize=3, label="Validation Accuracy")
axes[1].set_title("Validation Accuracy Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
print("  Saved → training_curves.png")
plt.show()

In [ ]:
# 6. MODEL EVALUATION
print("\n" + "=" * 60)
print("  STEP 5: Model Evaluation on Test Set")
print("=" * 60)

y_pred = mlp.predict(X_test)
y_prob = mlp.predict_proba(X_test)[:, 1]   # Probability of default

acc  = accuracy_score(y_test,  y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test,    y_pred, zero_division=0)
f1   = f1_score(y_test,        y_pred, zero_division=0)

print(f"\n  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")

print("\n  Full Classification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["No Default", "Default"]
))



In [ ]:
# 7. CONFUSION MATRIX

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["No Default", "Default"],
    yticklabels=["No Default", "Default"],
    annot_kws={"size": 14},
)
ax.set_title("Confusion Matrix – MLP Loan Default Prediction",
             fontweight="bold", pad=15)
ax.set_ylabel("Actual Label", fontsize=12)
ax.set_xlabel("Predicted Label", fontsize=12)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
print("\n  Saved → confusion_matrix.png")
plt.show()



In [ ]:
# 8. FEATURE IMPORTANCE (first-layer weight magnitudes)
feature_names = num_cols + cat_cols
# Mean absolute weight across all neurons in the first hidden layer
importance = np.abs(mlp.coefs_[0]).mean(axis=1)
feat_imp = (
    pd.Series(importance, index=feature_names)
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_title(
    "Feature Importance (Mean |Weight| – First Hidden Layer)",
    fontweight="bold"
)
ax.set_xlabel("Mean Absolute Weight")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
print("  Saved → feature_importance.png")
plt.show()

In [ ]:
# 9. SUMMARY
print("\n" + "=" * 60)
print("  SUMMARY")
print("=" * 60)
print(f"  Model      : MLP  (64-ReLU → 32-ReLU → 1-Sigmoid)")
print(f"  Optimizer  : Adam (lr=0.001) | Loss: Binary Cross-Entropy")
print(f"  Epochs ran : {mlp.n_iter_}")
print(f"  Accuracy   : {acc*100:.2f}%")
print(f"  Precision  : {prec*100:.2f}%")
print(f"  Recall     : {rec*100:.2f}%")
print(f"  F1-Score   : {f1*100:.2f}%")
print("=" * 60)
print("  Plots saved: training_curves.png | confusion_matrix.png")
print("               feature_importance.png")
print("=" * 60)
print("\n  ✓ Done.")
